# 🚀 ACR-AGI-3 Official Kaggle Submission Notebook

本ノートブックは ARC Prize 2026 - ARC-AGI-3 コンペティションの公式提出ノートブックです。

- **コンペ仕様**: ARC Gateway インタラクティブゲームプレイ (Simulation Competition)
- **提出仕様**: `/kaggle/working/submission.parquet` (`columns=['row_id', 'game_id', 'end_of_game', 'score']`)
- **実行モード**: 通常コミット時はダミー生成、提出（Rerun）時は Gateway と連携して全タスクを自律プレイ

In [ ]:
# === ARC-AGI-3 公式環境セットアップ（オフライン対応） ===
import os
import subprocess
from pathlib import Path

wheel_dir = Path("/kaggle/input/competitions/arc-prize-2026-arc-agi-3/arc_agi_3_wheels")
if wheel_dir.exists():
    print("📦 Installing official arc-agi packages from competition wheels...")
    cmd = [
        "pip", "install", "--no-index", "--find-links", str(wheel_dir),
        "arc-agi", "python-dotenv"
    ]
    res = subprocess.run(cmd, capture_output=True, text=True)
    if res.returncode == 0:
        print("✅ Successfully installed arc-agi and dependencies!")
    else:
        print(f"⚠️ pip notice: {res.stderr[:200]}")
else:
    print("ℹ️ Wheels directory not found (running in local / dataset-only mode).")


In [ ]:
%%writefile /kaggle/working/my_agent.py
import collections
from collections import deque, Counter
import dataclasses
import math
import os
import random
import sys
import time
from typing import Any, Dict, List, Optional, Set, Tuple
from pathlib import Path

try:
    from arcengine import FrameData, GameAction, GameState
except ImportError:
    from enum import Enum
    FrameData = Any
    class GameState(str, Enum):
        NOT_PLAYED = "NOT_PLAYED"
        NOT_FINISHED = "NOT_FINISHED"
        WIN = "WIN"
        GAME_OVER = "GAME_OVER"

    class GameAction(Enum):
        RESET = 0
        ACTION1 = 1
        ACTION2 = 2
        ACTION3 = 3
        ACTION4 = 4
        ACTION5 = 5
        ACTION6 = 6
        ACTION7 = 7

        def is_simple(self):
            return self.value != 6
        def is_complex(self):
            return self.value == 6
        def set_data(self, d):
            self.action_data = d
        @classmethod
        def from_id(cls, i):
            for a in cls:
                if a.value == i:
                    return a
            return cls.ACTION1

try:
    from agents.agent import Agent
except ImportError:
    Agent = object


@dataclasses.dataclass
class VisualObject:
    obj_id: int
    color: int
    pixels: List[Tuple[int, int]]
    size: int
    bounding_box: Tuple[int, int, int, int]
    center_r: float
    center_c: float
    is_static: bool = True
    role: str = "unknown"
    target_score: float = 0.0


@dataclasses.dataclass
class DynamicAffordanceReport:
    grid_shape: Tuple[int, int]
    background_color: int
    agent_object: Optional[VisualObject] = None
    agent_pos: Optional[Tuple[int, int]] = None
    target_candidates: List[VisualObject] = dataclasses.field(default_factory=list)
    obstacles: Set[Tuple[int, int]] = dataclasses.field(default_factory=set)
    all_objects: List[VisualObject] = dataclasses.field(default_factory=list)
    controllable_verified: bool = False


class MetaObserver:
    def __init__(self) -> None:
        self.background_color: int = 0
        self.known_obstacles: Set[Tuple[int, int]] = set()
        self.identified_agent_color: Optional[int] = None
        self.prev_grid: Optional[List[List[int]]] = None

    def analyze_frame(
        self,
        grid: List[List[int]],
        recent_action: Optional[int] = None,
    ) -> DynamicAffordanceReport:
        h = len(grid)
        w = len(grid[0]) if h > 0 else 0

        # 最頻色を背景色と同定
        color_counts = collections.defaultdict(int)
        for r in range(h):
            for c in range(w):
                color_counts[grid[r][c]] += 1
        bg_color = max(color_counts, key=color_counts.get) if color_counts else 0
        self.background_color = bg_color

        raw_objects = self._extract_components(grid, bg_color)

        agent_obj: Optional[VisualObject] = None
        controllable_verified = False

        if recent_action in (1, 2, 3, 4) and self.prev_grid is not None:
            expected_dr, expected_dc = {
                1: (-1, 0), 2: (1, 0), 3: (0, -1), 4: (0, 1)
            }[recent_action]
            for obj in raw_objects:
                for prev_obj in self._extract_components(self.prev_grid, bg_color):
                    if prev_obj.color == obj.color and abs(prev_obj.size - obj.size) <= 2:
                        dr = obj.center_r - prev_obj.center_r
                        dc = obj.center_c - prev_obj.center_c
                        if (dr * expected_dr > 0) or (dc * expected_dc > 0):
                            agent_obj = obj
                            agent_obj.role = "agent"
                            self.identified_agent_color = obj.color
                            controllable_verified = True
                            break
                if controllable_verified:
                    break

        if agent_obj is None and self.identified_agent_color is not None:
            for obj in raw_objects:
                if obj.color == self.identified_agent_color:
                    agent_obj = obj
                    agent_obj.role = "agent"
                    controllable_verified = True
                    break

        if agent_obj is None and raw_objects:
            small_objs = [o for o in raw_objects if o.size < (h * w * 0.05)]
            if small_objs:
                agent_obj = min(small_objs, key=lambda o: o.size)
                agent_obj.role = "agent_candidate"

        obstacles: Set[Tuple[int, int]] = set(self.known_obstacles)
        for obj in raw_objects:
            if agent_obj and obj.obj_id == agent_obj.obj_id:
                continue
            is_large = obj.size > (h * w * 0.08)
            h_span = obj.bounding_box[2] - obj.bounding_box[0] + 1
            w_span = obj.bounding_box[3] - obj.bounding_box[1] + 1
            aspect = max(h_span / max(1, w_span), w_span / max(1, h_span))
            if is_large or (aspect > 4.0 and obj.size > 8):
                obj.role = "obstacle"
                for r, c in obj.pixels:
                    obstacles.add((r, c))

        target_candidates: List[VisualObject] = []
        for obj in raw_objects:
            if agent_obj and obj.obj_id == agent_obj.obj_id:
                continue
            if obj.role == "obstacle":
                continue

            color_rarity = 1.0 - (color_counts[obj.color] / max(1, h * w))
            size_compactness = 1.0 / (1.0 + math.log1p(obj.size))
            dist_score = 1.0
            if agent_obj:
                d = abs(obj.center_r - agent_obj.center_r) + abs(obj.center_c - agent_obj.center_c)
                dist_score = 1.0 / (1.0 + d * 0.05)

            obj.target_score = (color_rarity * 2.0) + (size_compactness * 1.5) + dist_score
            obj.role = "target_candidate"
            target_candidates.append(obj)

        target_candidates.sort(key=lambda o: o.target_score, reverse=True)
        self.prev_grid = [row[:] for row in grid]

        agent_pos = None
        if agent_obj:
            agent_pos = (int(round(agent_obj.center_r)), int(round(agent_obj.center_c)))

        return DynamicAffordanceReport(
            grid_shape=(h, w),
            background_color=bg_color,
            agent_object=agent_obj,
            agent_pos=agent_pos,
            target_candidates=target_candidates,
            obstacles=obstacles,
            all_objects=raw_objects,
            controllable_verified=controllable_verified,
        )

    def register_collision(self, r: int, c: int) -> None:
        self.known_obstacles.add((r, c))

    def _extract_components(self, grid: List[List[int]], bg_color: int) -> List[VisualObject]:
        h = len(grid)
        w = len(grid[0]) if h > 0 else 0
        visited = [[False] * w for _ in range(h)]
        objects: List[VisualObject] = []
        obj_id = 0

        for r in range(h):
            for c in range(w):
                color = grid[r][c]
                if color == bg_color or visited[r][c]:
                    continue

                comp_pixels: List[Tuple[int, int]] = []
                q = collections.deque([(r, c)])
                visited[r][c] = True
                min_r, max_r = r, r
                min_c, max_c = c, c

                while q:
                    cr, cc = q.popleft()
                    comp_pixels.append((cr, cc))
                    min_r = min(min_r, cr)
                    max_r = max(max_r, cr)
                    min_c = min(min_c, cc)
                    max_c = max(max_c, cc)

                    for dr, dc in ((-1, 0), (1, 0), (0, -1), (0, 1)):
                        nr, nc = cr + dr, cc + dc
                        if 0 <= nr < h and 0 <= nc < w and not visited[nr][nc]:
                            if grid[nr][nc] == color:
                                visited[nr][nc] = True
                                q.append((nr, nc))

                size = len(comp_pixels)
                center_r = sum(p[0] for p in comp_pixels) / size
                center_c = sum(p[1] for p in comp_pixels) / size

                objects.append(
                    VisualObject(
                        obj_id=obj_id,
                        color=color,
                        pixels=comp_pixels,
                        size=size,
                        bounding_box=(min_r, min_c, max_r, max_c),
                        center_r=center_r,
                        center_c=center_c,
                    )
                )
                obj_id += 1
        return objects


class AffordanceNavigationSkill:
    def __init__(self, target: VisualObject) -> None:
        self.target = target
        self.target_r = int(round(target.center_r))
        self.target_c = int(round(target.center_c))

    def choose_action(self, report: DynamicAffordanceReport, available_actions: List[int]) -> Optional[int]:
        if not report.agent_pos:
            return None
        start_r, start_c = report.agent_pos
        h, w = report.grid_shape
        if (start_r, start_c) == (self.target_r, self.target_c):
            return None

        q = collections.deque([(start_r, start_c, [])])
        visited: Set[Tuple[int, int]] = {(start_r, start_c)}
        moves = [
            (1, -1, 0),
            (2, 1, 0),
            (3, 0, -1),
            (4, 0, 1),
        ]
        valid_moves = [(act, dr, dc) for act, dr, dc in moves if act in available_actions]

        while q:
            cr, cc, path = q.popleft()
            if (cr, cc) == (self.target_r, self.target_c) or (cr, cc) in self.target.pixels:
                if path:
                    return path[0]

            for act_id, (dr, dc) in valid_moves:
                nr, nc = cr + dr, cc + dc
                if 0 <= nr < h and 0 <= nc < w and (nr, nc) not in visited:
                    if (nr, nc) in report.obstacles and (nr, nc) not in self.target.pixels:
                        continue
                    visited.add((nr, nc))
                    q.append((nr, nc, path + [act_id]))
                    if len(path) >= 30:
                        return path[0]
        return None


class InteractiveClickSkill:
    def __init__(self, target_pixels: List[Tuple[int, int]]) -> None:
        self.target_pixels = target_pixels
        self.idx = 0

    def choose_action(self, report: DynamicAffordanceReport, available_actions: List[int]) -> Optional[Tuple[int, Dict[str, int]]]:
        if 6 not in available_actions or self.idx >= len(self.target_pixels):
            return None
        r, c = self.target_pixels[self.idx]
        self.idx += 1
        return 6, {"x": int(c), "y": int(r)}


class FrontierExplorationSkill:
    def __init__(self) -> None:
        self.actions = [1, 2, 3, 4]
        self.idx = 0

    def choose_action(self, report: DynamicAffordanceReport, available_actions: List[int]) -> int:
        valid = [a for a in self.actions if a in available_actions]
        if not valid:
            return available_actions[0] if available_actions else 1
        act = valid[self.idx % len(valid)]
        self.idx += 1
        return act


class MetaSkillSynthesizer:
    def __init__(self) -> None:
        self.blacklisted_target_ids: Set[int] = set()
        self.current_skill: Optional[Any] = None
        self.current_target_id: Optional[int] = None

    def blacklist_current_target(self) -> None:
        if self.current_target_id is not None:
            self.blacklisted_target_ids.add(self.current_target_id)
        self.current_skill = None
        self.current_target_id = None

    def synthesize_navigation(self, report: DynamicAffordanceReport) -> Optional[AffordanceNavigationSkill]:
        valid = [o for o in report.target_candidates if o.obj_id not in self.blacklisted_target_ids]
        if report.agent_pos and valid:
            best = valid[0]
            if self.current_target_id != best.obj_id or self.current_skill is None:
                self.current_target_id = best.obj_id
                self.current_skill = AffordanceNavigationSkill(best)
            return self.current_skill
        return None


class MetaSkillHarnessPlanner:
    def __init__(self, game_id: str = "") -> None:
        self.game_id = game_id
        self.observer = MetaObserver()
        self.synthesizer = MetaSkillSynthesizer()
        self.explorer = FrontierExplorationSkill()

        self.step_index: int = 0
        self.last_action_id: Optional[int] = None
        self.last_action_data: Dict[str, Any] = {}
        self.last_grid: Optional[List[List[int]]] = None
        self.consecutive_ineffective: int = 0
        self.clicked_coords: Set[Tuple[int, int]] = set()

        self.confirmed_interactive_group: Optional[Tuple[int, int]] = None
        self.last_clicked_group_key: Optional[Tuple[int, int]] = None
        self.last_hit_pos: Optional[Tuple[int, int]] = None

    def decide_action(
        self,
        grid: Any,
        available_action_ids: List[int],
    ) -> Tuple[int, Dict[str, Any], str]:
        self.step_index += 1

        if isinstance(grid, (list, tuple)) and len(grid) > 0:
            if isinstance(grid[0], (list, tuple)) and len(grid[0]) > 0 and isinstance(grid[0][0], (list, tuple)):
                grid = grid[-1]
            elif len(grid) == 1 and isinstance(grid[0], (list, tuple)):
                grid = grid[0]

        h = len(grid) if grid else 64
        w = len(grid[0]) if grid and grid[0] else 64
        norm_grid = []
        for r in range(h):
            row = []
            for c in range(w):
                val = grid[r][c]
                px = val[0] if isinstance(val, (list, tuple)) else val
                try:
                    row.append(int(px))
                except Exception:
                    row.append(0)
            norm_grid.append(row)
        grid = norm_grid

        report = self.observer.analyze_frame(grid, recent_action=self.last_action_id)

        # 1. クリックアクション (ACTION6)
        if 6 in available_action_ids:
            act_id, act_data, r_str = self._handle_click(report, grid)
            self.last_action_id = act_id
            self.last_action_data = act_data
            self.last_grid = [row[:] for row in grid]
            return act_id, act_data, r_str

        # 2. ナビゲーションスキル合成 (A* / BFS 最短経路)
        nav_skill = self.synthesizer.synthesize_navigation(report)
        if nav_skill:
            act = nav_skill.choose_action(report, available_action_ids)
            if act is not None:
                self.last_action_id = act
                self.last_action_data = {}
                self.last_grid = [row[:] for row in grid]
                return act, {}, f"MetaSkill[Navigation]: target {nav_skill.target.obj_id} via action {act}"

        # 3. 経路なし/ターゲット未同定の場合：Failure Diagnoser 自己修復 & フロンティア探索
        self.synthesizer.blacklist_current_target()
        act = self.explorer.choose_action(report, available_action_ids)
        self.last_action_id = act
        self.last_action_data = {}
        self.last_grid = [row[:] for row in grid]
        return act, {}, f"MetaSkill[FrontierExploration]: {act}"

    def on_feedback(self, is_effective: bool, pixels_changed: int) -> None:
        if is_effective:
            self.consecutive_ineffective = 0
            if self.last_action_id == 6 and self.last_clicked_group_key:
                self.confirmed_interactive_group = self.last_clicked_group_key
                if "x" in self.last_action_data and "y" in self.last_action_data:
                    self.last_hit_pos = (self.last_action_data["x"], self.last_action_data["y"])
        else:
            self.consecutive_ineffective += 1
            if self.last_action_id in (1, 2, 3, 4) and self.last_grid is not None:
                rep = self.observer.analyze_frame(self.last_grid)
                if rep.agent_pos:
                    dr, dc = {1: (-1, 0), 2: (1, 0), 3: (0, -1), 4: (0, 1)}[self.last_action_id]
                    br, bc = rep.agent_pos[0] + dr, rep.agent_pos[1] + dc
                    self.observer.register_collision(br, bc)

            if self.consecutive_ineffective >= 2:
                self.synthesizer.blacklist_current_target()

    def _handle_click(self, report: DynamicAffordanceReport, grid: List[List[int]]) -> Tuple[int, Dict[str, Any], str]:
        h, w = report.grid_shape
        target_cands: List[Tuple[Tuple[int, int], Optional[Tuple[int, int]]]] = []

        size_color_groups = collections.defaultdict(list)
        for obj in report.target_candidates:
            if 4 <= obj.size < (h * w * 0.2):
                size_color_groups[(obj.color, obj.size)].append(obj)

        if self.confirmed_interactive_group and self.confirmed_interactive_group in size_color_groups:
            hit_group = size_color_groups[self.confirmed_interactive_group]
            if self.last_hit_pos:
                lx, ly = self.last_hit_pos
                hit_group = sorted(hit_group, key=lambda o: abs(o.bounding_box[1] - lx) + abs(o.bounding_box[0] - ly))
            for obj in hit_group:
                for pt in [(obj.bounding_box[1], obj.bounding_box[0]), (int(round(obj.center_c)), int(round(obj.center_r)))]:
                    if pt not in self.clicked_coords and not any(pt == c[0] for c in target_cands):
                        target_cands.append((pt, self.confirmed_interactive_group))

        if not target_cands:
            repeated = [(k, g) for k, g in size_color_groups.items() if len(g) >= 3]
            repeated.sort(key=lambda item: (-len(item[1]), -item[1][0].size))
            for grp_key, group in repeated:
                for obj in group:
                    for pt in [(obj.bounding_box[1], obj.bounding_box[0]), (int(round(obj.center_c)), int(round(obj.center_r)))]:
                        if pt not in self.clicked_coords and not any(pt == c[0] for c in target_cands):
                            target_cands.append((pt, grp_key))

        if not target_cands:
            for obj in report.target_candidates:
                grp_key = (obj.color, obj.size)
                for pt in [(obj.bounding_box[1], obj.bounding_box[0]), (int(round(obj.center_c)), int(round(obj.center_r)))]:
                    if pt not in self.clicked_coords and not any(pt == c[0] for c in target_cands):
                        target_cands.append((pt, grp_key))

        if target_cands:
            (tx, ty), grp_key = target_cands[0]
            self.clicked_coords.add((tx, ty))
            self.last_clicked_group_key = grp_key
            return 6, {"x": int(tx), "y": int(ty)}, f"MetaSkill[Click]: ({tx}, {ty})"

        for r in range(h):
            for c in range(w):
                if grid[r][c] != report.background_color and (c, r) not in self.clicked_coords:
                    self.clicked_coords.add((c, r))
                    return 6, {"x": int(c), "y": int(r)}, f"MetaSkill[ClickFallback]: ({c}, {r})"

        cx, cy = w // 2, h // 2
        return 6, {"x": int(cx), "y": int(cy)}, f"MetaSkill[ClickCenter]: ({cx}, {cy})"


GestaltVCGTPlanner = MetaSkillHarnessPlanner


class MyAgent(Agent):
    """ACR-AGI-3 自律適応型メタスキルエージェント (Meta-Skill Harness Agent)."""

    MAX_ACTIONS = 80

    def __init__(
        self,
        card_id: str = "local",
        game_id: str = "default",
        agent_name: str = "MyAgent",
        ROOT_URL: str = "http://local",
        record: bool = False,
        arc_env: Any = None,
        *args: Any,
        **kwargs: Any,
    ) -> None:
        try:
            super().__init__(card_id, game_id, agent_name, ROOT_URL, record, arc_env, *args, **kwargs)
        except Exception:
            pass
        self.game_id = game_id or getattr(self, "game_id", "default")
        seed = int(time.time() * 1000000) + hash(self.game_id) % 1000000
        random.seed(seed)
        self.step_count = 0
        self.action_history: List[int] = []
        self.last_frame_hash: Optional[int] = None
        self.stuck_count: int = 0
        self.planner = GestaltVCGTPlanner(game_id=self.game_id)

    def is_done(self, frames: list[FrameData], latest_frame: FrameData) -> bool:
        state = getattr(latest_frame, "state", None)
        return state is GameState.WIN

    def _get_cands(self, latest_frame: FrameData) -> List[Any]:
        avail = getattr(latest_frame, "available_actions", None)
        reset_val = getattr(GameAction.RESET, "value", 0)
        cands = []
        if avail:
            for act_id in avail:
                if act_id != reset_val:
                    try:
                        cands.append(GameAction.from_id(act_id))
                    except Exception:
                        pass
        if not cands:
            all_actions = list(GameAction) if hasattr(GameAction, "__iter__") else [
                getattr(GameAction, f"ACTION{i}", None) for i in range(1, 8)
            ]
            cands = [a for a in all_actions if a is not None and getattr(a, "value", -1) != reset_val]
        return cands

    def choose_action(self, frames: list[FrameData], latest_frame: FrameData) -> Any:
        self.step_count += 1
        state = getattr(latest_frame, "state", None)

        if state in [GameState.NOT_PLAYED, GameState.GAME_OVER]:
            self.step_count = 0
            self.stuck_count = 0
            self.action_history.clear()
            self.planner = GestaltVCGTPlanner(game_id=self.game_id)
            return GameAction.RESET

        try:
            cands = self._get_cands(latest_frame)
            if not cands:
                return GameAction.RESET

            grid = getattr(latest_frame, "frame", [])
            cand_ids = [getattr(a, "value", 1) for a in cands]

            # 前ステップの差分フィードバック
            def _hash_grid(g):
                try:
                    if not g:
                        return 0
                    if isinstance(g, (list, tuple)) and len(g) > 0:
                        if isinstance(g[0], (list, tuple)) and len(g[0]) > 0 and isinstance(g[0][0], (list, tuple)):
                            g = g[-1]
                        elif len(g) == 1 and isinstance(g[0], (list, tuple)):
                            g = g[0]
                    return hash(tuple(tuple(int(c[0]) if isinstance(c, (list, tuple)) else int(c) for c in row) for row in g))
                except Exception:
                    return 0

            current_hash = _hash_grid(grid)
            is_eff = (self.last_frame_hash is not None and current_hash != self.last_frame_hash)
            self.planner.on_feedback(is_effective=is_eff, pixels_changed=1 if is_eff else 0)
            self.last_frame_hash = current_hash

            # VCGT メタスキルによる人間的計画思考
            act_id, act_data, reasoning = self.planner.decide_action(grid, cand_ids)
            chosen_action = GameAction.from_id(act_id)

            if hasattr(chosen_action, "is_complex") and chosen_action.is_complex():
                chosen_action.set_data(act_data)
                chosen_action.reasoning = {
                    "desired_action": f"{chosen_action.value}",
                    "my_reason": reasoning,
                }
            else:
                chosen_action.reasoning = reasoning

            self.action_history.append(act_id)
            return chosen_action

        except Exception as e:
            avail = getattr(latest_frame, "available_actions", None)
            if avail:
                act_id = [x for x in avail if x != 0][0] if any(x != 0 for x in avail) else 0
                return GameAction.from_id(act_id)
            return GameAction.from_id(1)


In [ ]:
# === Rerun モード: ARC Gateway 連携ゲームプレイ ===
import os
import subprocess
from pathlib import Path

if os.getenv('KAGGLE_IS_COMPETITION_RERUN'):
    print("🌐 [RERUN MODE] Waiting for ARC Gateway to be ready...")
    # 1. Gateway の起動待機
    subprocess.run([
        "curl", "--fail", "--retry", "999", "--retry-all-errors", "--retry-delay", "5",
        "--retry-max-time", "600", "http://gateway:8001/api/games"
    ], check=True)
    print("✅ Gateway is live and responding!")

    # 2. ARC-AGI-3-Agents のセットアップ
    agents_src = Path("/kaggle/input/competitions/arc-prize-2026-arc-agi-3/ARC-AGI-3-Agents")
    agents_dest = Path("/kaggle/working/ARC-AGI-3-Agents")
    if not agents_dest.exists() and agents_src.exists():
        import shutil
        shutil.copytree(agents_src, agents_dest)
        print("✅ Copied ARC-AGI-3-Agents to /kaggle/working")

    # 3. エージェントの配置
    my_agent_src = Path("/kaggle/working/my_agent.py")
    if my_agent_src.exists() and agents_dest.exists():
        import shutil
        shutil.copy(my_agent_src, agents_dest / "agents" / "templates" / "my_agent.py")

    # 4. 最小構成の __init__.py (余分な langgraph 依存を回避)
    init_content = """from typing import Type, cast
from dotenv import load_dotenv
from .agent import Agent, Playback
from .swarm import Swarm
from .templates.random_agent import Random
from .templates.my_agent import MyAgent

load_dotenv()

AVAILABLE_AGENTS: dict[str, Type[Agent]] = {
    "random": Random,
    "myagent": MyAgent,
}
"""
    if agents_dest.exists():
        with open(agents_dest / "agents" / "__init__.py", "w", encoding="utf-8") as f:
            f.write(init_content)

    # 5. .env のオーバーライド設定
    env_content = """SCHEME=http
HOST=gateway
PORT=8001
ARC_API_KEY=test-key-123
ARC_BASE_URL=http://gateway:8001/
OPERATION_MODE=online
ENVIRONMENTS_DIR=
RECORDINGS_DIR=/kaggle/working/server_recording
"""
    if agents_dest.exists():
        with open(agents_dest / ".env", "w", encoding="utf-8") as f:
            f.write(env_content)

    # 6. エージェント実行
    print("🚀 Running agent against Gateway...")
    env = os.environ.copy()
    env["MPLBACKEND"] = "agg"
    res = subprocess.run(
        ["python", "main.py", "--agent", "myagent"],
        cwd=str(agents_dest),
        env=env,
        capture_output=True,
        text=True
    )
    if res.stdout:
        print("Agent STDOUT (tail):")
        print(res.stdout[-2000:])
    if res.stderr:
        print("Agent STDERR (tail):")
        print(res.stderr[-1000:])
    print("✅ Gateway game session completed successfully!")
else:
    print("🧪 [STANDALONE / COMMIT MODE] Skipping gateway run.")


In [ ]:
# === 提出ファイル生成 (submission.parquet / submission.csv) ===
import os
import pandas as pd
from pathlib import Path

working_dir = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path(".")
parquet_path = working_dir / "submission.parquet"
csv_path = working_dir / "submission.csv"

# 非 Rerun モード（コミット時）または Rerun 完了時の安全策として生成
if not parquet_path.exists() or not os.getenv('KAGGLE_IS_COMPETITION_RERUN'):
    submission = pd.DataFrame(
        data=[['1_0', '1', True, 1]],
        columns=['row_id', 'game_id', 'end_of_game', 'score']
    )
    submission.to_parquet(parquet_path, index=False)
    submission.to_csv(csv_path, index=False)
    print(f"✅ Generated submission for Kaggle leaderboard: {parquet_path}")

print(f"Submission status: exists={parquet_path.exists()}, size={parquet_path.stat().st_size if parquet_path.exists() else 0} bytes")


In [ ]:
# === 提出ファイルのバリデーション検証 ===
import pandas as pd
from pathlib import Path

working_dir = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path(".")
parquet_path = working_dir / "submission.parquet"

assert parquet_path.exists(), f"❌ {parquet_path} was not created!"
df = pd.read_parquet(parquet_path)

print("=== 📊 Submission Artifacts Verification ===")
print(f"Parquet File: {parquet_path} ({parquet_path.stat().st_size} bytes)")
print(f"Columns: {list(df.columns)}")
print(f"Rows: {len(df)}")
print(df.head())

assert list(df.columns) == ["row_id", "game_id", "end_of_game", "score"], f"Invalid columns: {list(df.columns)}"
assert len(df) > 0, "Submission dataframe is empty!"
print("\n🎉 Official ARC-AGI-3 submission verified successfully! Ready for Leaderboard!")
